# Sentinel Auth — corrected RBA benchmark

Thí nghiệm này kiểm chứng đúng sáu feature cố định của Sentinel trên 10 triệu login đầu của RBA. Mọi feature chỉ dùng lịch sử trước sự kiện hiện tại; nhãn chỉ dùng cho đánh giá và Logistic Regression `DIAGNOSTIC_ONLY`.

Nguồn: [DAS Group RBA dataset](https://github.com/das-group/rba-dataset), [RBA algorithm](https://github.com/das-group/rba-algorithm), [Wiefling et al. 2022](https://doi.org/10.1145/3546069), và [scikit-learn](https://scikit-learn.org/stable/modules/outlier_detection.html). RBA là dữ liệu tổng hợp nghiên cứu; artifact luôn bắt đầu ở trạng thái `experimental`.

## 00 — Cấu hình và tái lập

Baseline trước forensic được giữ nguyên trong `BASELINE_10M`. Semantics sửa đổi không thay tên, kiểu hoặc miền giá trị của sáu feature.

In [ ]:
from collections import Counter, deque
from datetime import datetime, timezone
from pathlib import Path
from threading import Event, Thread
from time import perf_counter
import gc
import json
import platform

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression, SGDOneClassSVM
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_recall_curve,
    precision_recall_fscore_support, roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler

def progress(message):
    print(f'[{datetime.now(timezone.utc).isoformat()}] {message}', flush=True)

heartbeat_stop = Event()
def heartbeat():
    while not heartbeat_stop.wait(60):
        progress('heartbeat: notebook active')
Thread(target=heartbeat, daemon=True).start()

DATA_PATH = Path('/kaggle/input/datasets/dasgroup/rba-dataset/rba-dataset.csv')
OUTPUT_DIR = Path('/kaggle/working/sentinel-rba')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NROWS = 10_000_000
TRAIN_RATIO = 0.80
RANDOM_STATE = 42
MODEL_VERSION = 'v1.1-isolation-forest-rba-corrected-experimental'
FEATURE_SEMANTICS_VERSION = 'sentinel-six-v1.1'
FEATURES = [
    'hour_of_day', 'fail_count_24h', 'ip_change_rate_7d', 'new_device',
    'average_login_interval_seconds', 'deviation_score',
]
TARGETS = ['Is Account Takeover', 'Is Attack IP']
BASELINE_10M = {
    'rows': 10_000_000, 'users': 2_075_123, 'train_rows': 8_000_000,
    'test_rows': 2_000_000, 'cutoff_utc': '2020-05-23T13:12:33.886000+00:00',
    'test_ato': 14, 'test_attack_ip': 185_355,
    'ato_if_ap': 0.000011, 'ato_if_ap_lift': 1.58, 'ato_if_roc_auc': 0.516,
    'ato_if_recall': 0.571, 'ato_if_flag_rate': 0.3676,
    'attack_ip_if_ap': 0.074625, 'attack_ip_if_roc_auc': 0.431,
    'if_temporal_roc_auc': [0.787, 0.457, 0.335, 0.538],
}
progress(f'python={platform.python_version()} sklearn={sklearn.__version__} rows={NROWS:,}')

## 01 — Đọc RBA

Chỉ đọc các trường cần để dựng sáu feature và hai nhãn. RTT, vị trí và ASN hợp lệ trong RBA nhưng nằm ngoài contract Sentinel.

In [ ]:
progress('load: start')
columns = [
    'User ID', 'Login Timestamp', 'IP Address', 'User Agent String',
    'Login Successful', *TARGETS,
]
df = pd.read_csv(DATA_PATH, usecols=columns, nrows=NROWS, low_memory=False)
progress(f'load: complete shape={df.shape} memory_mib={df.memory_usage(deep=True).sum()/2**20:.1f}')
df.head()

## 02 — Kiểm tra nguồn và nhãn

Các dòng thiếu trường bắt buộc bị loại trước feature engineering và số lượng bị loại được in rõ. Không dùng nhãn để chọn cutoff.

In [ ]:
progress('validation: start')
source_missing = df[columns].isna().sum().rename('missing')
df['Login Timestamp'] = pd.to_datetime(df['Login Timestamp'], utc=True, errors='coerce')
for column in ['Login Successful', *TARGETS]:
    if not pd.api.types.is_bool_dtype(df[column]):
        df[column] = (df[column].astype('string').str.lower()
                      .map({'true': True, 'false': False, '1': True, '0': False}))
before_drop = len(df)
df = df.dropna(subset=columns)
print(source_missing)
print('dropped rows:', before_drop - len(df))
print('label positives:', {target: int(df[target].sum()) for target in TARGETS})
progress('validation: complete')

## 03 — Lịch sử theo thời gian

Sắp xếp ổn định theo user và timestamp. Mọi cửa sổ sau đây đều là nửa kín `[lower_bound, current_timestamp)`.

In [ ]:
progress('chronological sort: start')
df = df.sort_values(['User ID', 'Login Timestamp'], kind='mergesort').reset_index(drop=True)
assert df.groupby('User ID', sort=False)['Login Timestamp'].apply(
    lambda values: values.is_monotonic_increasing).all()
print(f'rows: {len(df):,} | users: {df["User ID"].nunique():,}')
progress('chronological sort: complete')

## 04 — Sáu feature đã hiệu chỉnh

- `hour_of_day`: giờ UTC của event (`PROJECT_DEFINED_CONTRACT_CLARIFICATION`).
- `fail_count_24h`: lỗi của cùng user trong `[t-24h,t)`.
- `ip_change_rate_7d`: `1 - count(current IP)/count(prior logins)` trong `[t-7d,t)`; không lịch sử = 0 (`PROJECT_DEFINED_RBA_ADAPTATION`).
- `new_device`: exact User-Agent proxy chưa thấy trong `[t-30d,t)`; UA thiếu = false (`PROJECT_DEFINED_UA_PROXY`).
- `average_login_interval_seconds`: trung bình tối đa 9 khoảng cách từ 10 timestamp trước; thiếu lịch sử = 0 (`KEEP_PROJECT_DEFINED`).
- `deviation_score`: độ lệch tương đối của current gap so với trung bình trên, clip `[0,1]`; thiếu lịch sử = 0 (`KEEP_PROJECT_DEFINED`).

In [ ]:
progress('vector features: start')
df['hour_of_day'] = df['Login Timestamp'].dt.hour.astype('int8')
df['_failed'] = (~df['Login Successful']).astype('int8')
df['fail_count_24h'] = (
    df.groupby('User ID', sort=False)
      .rolling('24h', on='Login Timestamp', closed='left')['_failed'].sum()
      .fillna(0).to_numpy().astype('int32')
)
df['_gap'] = df.groupby('User ID', sort=False)['Login Timestamp'].diff().dt.total_seconds()
df['average_login_interval_seconds'] = (
    df.groupby('User ID', sort=False)['_gap']
      .transform(lambda values: values.shift().rolling(9, min_periods=1).mean())
      .fillna(0).astype('int64')
)
historical_mean = df['average_login_interval_seconds']
df['deviation_score'] = (
    ((df['_gap'] - historical_mean).abs() / historical_mean.where(historical_mean.gt(0)))
    .fillna(0).clip(0, 1).astype('float32')
)
progress('vector features: complete')

In [ ]:
progress('IP/UA history: start')
ip_rate = np.zeros(len(df), dtype='float32')
new_device = np.zeros(len(df), dtype=bool)
current_user = object()
ip_queue, device_queue = deque(), deque()
ip_counts, device_counts = Counter(), Counter()

events = df[['User ID', 'Login Timestamp', 'IP Address', 'User Agent String']]
for row_no, (user, timestamp, ip, device) in enumerate(
        events.itertuples(index=False, name=None)):
    if user != current_user:
        current_user = user
        ip_queue.clear(); device_queue.clear(); ip_counts.clear(); device_counts.clear()
    while ip_queue and ip_queue[0][0] < timestamp - pd.Timedelta(days=7):
        _, expired = ip_queue.popleft()
        ip_counts[expired] -= 1
        if not ip_counts[expired]:
            del ip_counts[expired]
    while device_queue and device_queue[0][0] < timestamp - pd.Timedelta(days=30):
        _, expired = device_queue.popleft()
        device_counts[expired] -= 1
        if not device_counts[expired]:
            del device_counts[expired]

    prior_login_count = len(ip_queue)
    if prior_login_count and pd.notna(ip):
        ip_rate[row_no] = 1.0 - ip_counts.get(ip, 0) / prior_login_count
    if pd.notna(device):
        new_device[row_no] = device_counts.get(device, 0) == 0

    ip_queue.append((timestamp, ip)); ip_counts[ip] += 1
    device_queue.append((timestamp, device)); device_counts[device] += 1
    if (row_no + 1) % 500_000 == 0:
        progress(f'IP/UA history rows: {row_no + 1:,}/{len(df):,}')

df['ip_change_rate_7d'] = ip_rate
df['new_device'] = new_device
progress('IP/UA history: complete')

## 05 — Audit contract và parity semantics

Các assert kiểm tra contract, causal boundaries và cold-start; đây không phải tuning.

In [ ]:
progress('feature audit: start')
model_data = df[['User ID', 'Login Timestamp', *TARGETS, *FEATURES]].copy()
assert model_data[FEATURES].notna().all().all()
assert model_data['hour_of_day'].between(0, 23).all()
assert model_data['fail_count_24h'].ge(0).all()
assert model_data['ip_change_rate_7d'].between(0, 1).all()
assert model_data['new_device'].isin([True, False]).all()
assert model_data['average_login_interval_seconds'].ge(0).all()
assert model_data['deviation_score'].between(0, 1).all()

fixture = pd.DataFrame({
    'User ID': ['fixture-user'] * 3,
    'Login Timestamp': pd.to_datetime(['2026-02-01T11:50:00Z', '2026-02-01T11:55:00Z', '2026-02-01T12:00:00Z']),
    'IP Address': ['a', 'b', 'new'], 'User Agent String': ['known', 'known', 'new'],
    'Login Successful': [False, True, False],
})
def fixture_history(group):
    ipq, devq, ipc, devc, rows = deque(), deque(), Counter(), Counter(), []
    for timestamp, ip, device in group.itertuples(index=False, name=None):
        prior = len(ipq)
        rows.append((0.0 if not prior else 1 - ipc.get(ip, 0) / prior, devc.get(device, 0) == 0))
        ipq.append((timestamp, ip)); ipc[ip] += 1
        devq.append((timestamp, device)); devc[device] += 1
    return rows
fixture['_failed'] = (~fixture['Login Successful']).astype('int8')
fixture['fail_count_24h'] = (
    fixture.rolling('24h', on='Login Timestamp', closed='left')['_failed'].sum().fillna(0)
)
fixture['_gap'] = fixture['Login Timestamp'].diff().dt.total_seconds()
fixture['average_login_interval_seconds'] = fixture['_gap'].shift().rolling(9, min_periods=1).mean().fillna(0)
fixture['deviation_score'] = (
    (fixture['_gap'] - fixture['average_login_interval_seconds']).abs()
    / fixture['average_login_interval_seconds'].where(fixture['average_login_interval_seconds'].gt(0))
).fillna(0).clip(0, 1)
fixture_ip_rate, fixture_new_device = fixture_history(
    fixture[['Login Timestamp', 'IP Address', 'User Agent String']])[-1]
fixture_features = {
    'hour_of_day': int(fixture.iloc[-1]['Login Timestamp'].hour),
    'fail_count_24h': int(fixture.iloc[-1]['fail_count_24h']),
    'ip_change_rate_7d': fixture_ip_rate,
    'new_device': fixture_new_device,
    'average_login_interval_seconds': int(fixture.iloc[-1]['average_login_interval_seconds']),
    'deviation_score': float(fixture.iloc[-1]['deviation_score']),
}
assert fixture_features == {
    'hour_of_day': 12, 'fail_count_24h': 1, 'ip_change_rate_7d': 1.0,
    'new_device': True, 'average_login_interval_seconds': 300, 'deviation_score': 0.0,
}
print('deterministic parity fixture:', fixture_features)
print(model_data[FEATURES].assign(new_device=model_data['new_device'].astype('int8')).describe().T)
del df, events, ip_rate, new_device
gc.collect()
progress('feature audit: complete')

## 06 — Chronological split

Giữ đúng population 10M và tỷ lệ 80/20 để so sánh trực tiếp baseline. Nhãn không tham gia chọn cutoff.

In [ ]:
progress('chronological split: start')
model_data = model_data.sort_values('Login Timestamp', kind='mergesort').reset_index(drop=True)
split_at = int(len(model_data) * TRAIN_RATIO)
cutoff = model_data.loc[split_at, 'Login Timestamp']
train_mask = model_data['Login Timestamp'] < cutoff
test_mask = ~train_mask
X_train = model_data.loc[train_mask, FEATURES].astype('float32')
X_test = model_data.loc[test_mask, FEATURES].astype('float32')
y_train = model_data.loc[train_mask, TARGETS].astype('int8')
y_test = model_data.loc[test_mask, TARGETS].astype('int8')
test_time = model_data.loc[test_mask, 'Login Timestamp'].reset_index(drop=True)
print('cutoff:', cutoff)
print(f'train: {len(X_train):,} | test: {len(X_test):,}')
print(pd.concat({'train': y_train.sum(), 'test': y_test.sum()}, axis=1))
assert list(X_train.columns) == FEATURES and list(X_test.columns) == FEATURES
progress('chronological split: complete')

## 07 — Feature-only forensic statistics

Xuất missing/zero rate, percentile tổng thể và theo từng nhãn; univariate AP/AUC chỉ tính cho feature có chiều rủi ro đơn điệu hợp lý. `hour_of_day` là chu kỳ và interval có thể hai chiều nên không ép một AUC gây hiểu lầm.

In [ ]:
progress('feature forensics: start')
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
def summarize(frame, scope, target='', label=''):
    rows = []
    for feature in FEATURES:
        values = frame[feature].astype(float)
        desc = values.describe(percentiles=percentiles)
        rows.append({
            'scope': scope, 'target': target, 'label': label, 'feature': feature,
            'count': int(desc['count']), 'missing_rate': float(values.isna().mean()),
            'zero_rate': float(values.eq(0).mean()), 'mean': float(desc['mean']),
            'std': float(desc['std']), 'min': float(desc['min']),
            'p01': float(desc['1%']), 'p05': float(desc['5%']),
            'p25': float(desc['25%']), 'p50': float(desc['50%']),
            'p75': float(desc['75%']), 'p95': float(desc['95%']),
            'p99': float(desc['99%']), 'max': float(desc['max']),
        })
    return rows

feature_rows = summarize(model_data, 'overall')
for target in TARGETS:
    for label in (0, 1):
        subset = model_data.loc[model_data[target].astype('int8').eq(label), FEATURES]
        feature_rows.extend(summarize(subset, 'label', target, str(label)))

monotonic_risk = ['fail_count_24h', 'ip_change_rate_7d', 'new_device', 'deviation_score']
univariate_rows = []
for target in TARGETS:
    truth = y_test[target]
    for feature in FEATURES:
        meaningful = feature in monotonic_risk and truth.nunique() == 2
        score = X_test[feature].astype(float)
        univariate_rows.append({
            'target': target, 'feature': feature,
            'average_precision': average_precision_score(truth, score) if meaningful else np.nan,
            'roc_auc': roc_auc_score(truth, score) if meaningful else np.nan,
            'note': '' if meaningful else 'not_monotonic_or_single_class',
        })

feature_statistics = pd.DataFrame(feature_rows)
univariate_statistics = pd.DataFrame(univariate_rows)
feature_statistics.to_csv(OUTPUT_DIR / 'feature_statistics.csv', index=False)
univariate_statistics.to_csv(OUTPUT_DIR / 'univariate_feature_metrics.csv', index=False)
display(feature_statistics)
display(univariate_statistics)
progress('feature forensics: complete')

In [ ]:
progress('monthly feature stability: start')
stability_features = ['ip_change_rate_7d', 'new_device', 'average_login_interval_seconds', 'deviation_score']
monthly_source = model_data[['Login Timestamp', *stability_features]].copy()
monthly_source['month'] = monthly_source['Login Timestamp'].dt.to_period('M').astype(str)
monthly_features = monthly_source.groupby('month', sort=True)[stability_features].agg(['mean', 'median'])
monthly_features.columns = [f'{feature}_{stat}' for feature, stat in monthly_features.columns]
monthly_features.to_csv(OUTPUT_DIR / 'feature_stability.csv')
display(monthly_features)
del monthly_source
gc.collect()
progress('monthly feature stability: complete')

## 08–09 — IsolationForest và diagnostic baselines

`B0` là no-skill; `B1` là ứng viên Sentinel; `B2` là baseline unsupervised scalable; `B3` dùng ATO label và chỉ có vai trò `DIAGNOSTIC_ONLY`.

In [ ]:
progress('model fitting: start')
models, fit_seconds = {}, {}

started = perf_counter()
models['B1 IsolationForest'] = IsolationForest(
    n_estimators=100, max_samples='auto', contamination='auto',
    random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_train)
fit_seconds['B1 IsolationForest'] = perf_counter() - started
progress(f'B1 complete seconds={fit_seconds["B1 IsolationForest"]:.2f}')

started = perf_counter()
models['B2 SGDOneClassSVM'] = make_pipeline(
    StandardScaler(), SGDOneClassSVM(random_state=RANDOM_STATE, nu=0.1),
).fit(X_train)
fit_seconds['B2 SGDOneClassSVM'] = perf_counter() - started
progress(f'B2 complete seconds={fit_seconds["B2 SGDOneClassSVM"]:.2f}')

started = perf_counter()
models['B3 LogisticRegression DIAGNOSTIC_ONLY'] = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight='balanced', max_iter=500, random_state=RANDOM_STATE),
).fit(X_train, y_train['Is Account Takeover'])
fit_seconds['B3 LogisticRegression DIAGNOSTIC_ONLY'] = perf_counter() - started
progress(f'B3 complete seconds={fit_seconds["B3 LogisticRegression DIAGNOSTIC_ONLY"]:.2f}')

## 10 — Metrics tổng thể

AP là metric chính. Không dùng accuracy làm headline. Với 14 ATO trong test, bootstrap CI được ghi `unavailable_too_few_positives` thay vì tạo khoảng tin cậy giả ổn định.

In [ ]:
def model_scores(name, model, X, prevalence):
    if name == 'B0 No-skill':
        return np.full(len(X), prevalence), np.zeros(len(X), dtype='int8')
    if name.startswith('B1') or name.startswith('B2'):
        return -model.decision_function(X), (model.predict(X) == -1).astype('int8')
    probability = model.predict_proba(X)[:, 1]
    return probability, (probability >= 0.5).astype('int8')

def evaluate(name, target, truth, rank_score, prediction, fit_time, window, start, end):
    prevalence = float(truth.mean())
    tn, fp, fn, tp = confusion_matrix(truth, prediction, labels=[0, 1]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        truth, prediction, average='binary', zero_division=0)
    valid = truth.nunique() == 2
    threshold_metrics = name != 'B0 No-skill'
    return {
        'model': name, 'target': target, 'window': window,
        'window_start_utc': start, 'window_end_utc': end,
        'test_rows': len(truth), 'positives': int(truth.sum()), 'prevalence': prevalence,
        'average_precision': average_precision_score(truth, rank_score) if valid else np.nan,
        'ap_lift': (average_precision_score(truth, rank_score) / prevalence)
                   if valid and prevalence else np.nan,
        'roc_auc': roc_auc_score(truth, rank_score) if valid else np.nan,
        'precision': precision if threshold_metrics else np.nan,
        'recall': recall if threshold_metrics else np.nan,
        'f1': f1 if threshold_metrics else np.nan,
        'flag_rate': float(prediction.mean()) if threshold_metrics else np.nan,
        'tn': int(tn) if threshold_metrics else np.nan,
        'fp': int(fp) if threshold_metrics else np.nan,
        'fn': int(fn) if threshold_metrics else np.nan,
        'tp': int(tp) if threshold_metrics else np.nan,
        'false_positives_per_1000': fp / len(truth) * 1_000 if threshold_metrics else np.nan,
        'false_positives_per_million': fp / len(truth) * 1_000_000 if threshold_metrics else np.nan,
        'fit_seconds': fit_time, 'ci_status': 'unavailable_too_few_positives' if target == TARGETS[0] else '',
    }

progress('overall scoring: start')
rows, cached = [], {}
for target in TARGETS:
    prevalence = float(y_train[target].mean())
    for name, model in [('B0 No-skill', None), *models.items()]:
        started = perf_counter()
        score_values, predictions = model_scores(name, model, X_test, prevalence)
        inference_seconds = perf_counter() - started
        row = evaluate(name, target, y_test[target], score_values, predictions,
                       0.0 if model is None else fit_seconds[name], 'overall',
                       test_time.iloc[0].isoformat(), test_time.iloc[-1].isoformat())
        row['inference_seconds'] = inference_seconds
        rows.append(row)
        cached[(name, target)] = (score_values, predictions)
benchmark = pd.DataFrame(rows)
benchmark.to_csv(OUTPUT_DIR / 'benchmark_metrics.csv', index=False)
display(benchmark)
progress('overall scoring: complete')

## 11 — Temporal out-of-time windows

Bốn cửa sổ liên tiếp đều xảy ra sau train cutoff. Mọi positive count được công bố; cửa sổ một lớp trả `NaN`.

In [ ]:
progress('temporal evaluation: start')
temporal_rows = []
for window_no, positions in enumerate(np.array_split(np.arange(len(X_test)), 4), start=1):
    for target in TARGETS:
        for name, model in [('B0 No-skill', None), *models.items()]:
            score_values, predictions = cached[(name, target)]
            row = evaluate(
                name, target, y_test[target].iloc[positions], score_values[positions],
                predictions[positions], 0.0 if model is None else fit_seconds[name],
                f'W{window_no}', test_time.iloc[positions[0]].isoformat(),
                test_time.iloc[positions[-1]].isoformat(),
            )
            row['inference_seconds'] = np.nan
            temporal_rows.append(row)
temporal_metrics = pd.DataFrame(temporal_rows)
temporal_metrics.to_csv(OUTPUT_DIR / 'temporal_metrics.csv', index=False)
display(temporal_metrics)
progress('temporal evaluation: complete')

## 12 — Error analysis

In [ ]:
progress('error analysis: start')
if_score, if_prediction = cached[('B1 IsolationForest', 'Is Account Takeover')]
error_analysis = model_data.loc[test_mask, ['User ID', 'Login Timestamp', *TARGETS, *FEATURES]].reset_index(drop=True)
error_analysis['if_anomaly_score'] = if_score
error_analysis['if_predicted_anomaly'] = if_prediction.astype(bool)
ato_errors = error_analysis.loc[
    error_analysis['Is Account Takeover'] | error_analysis['if_predicted_anomaly']
].sort_values('if_anomaly_score', ascending=False)
ato_errors.head(500).to_csv(OUTPUT_DIR / 'error_analysis_sample.csv', index=False)
print('flagged:', int(if_prediction.sum()))
print('ATO TP:', int(((if_prediction == 1) & (y_test[TARGETS[0]].to_numpy() == 1)).sum()))
print('ATO FN:', int(((if_prediction == 0) & (y_test[TARGETS[0]].to_numpy() == 1)).sum()))
progress('error analysis: complete')

## 13 — Biểu đồ phục vụ báo cáo

In [ ]:
progress('figures: start')
ato_truth = y_test['Is Account Takeover']

fig, ax = plt.subplots(figsize=(8, 5))
for name in models:
    scores, _ = cached[(name, 'Is Account Takeover')]
    precision, recall, _ = precision_recall_curve(ato_truth, scores)
    ax.plot(recall, precision, label=name)
ax.axhline(ato_truth.mean(), color='black', linestyle='--', label='No-skill prevalence')
ax.set(title='ATO precision–recall trên chronological test', xlabel='Recall', ylabel='Precision')
ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUTPUT_DIR / '01_ato_pr_curve.png', dpi=180); plt.close(fig)

ato_overall = benchmark[(benchmark.target == 'Is Account Takeover') & (benchmark.model != 'B0 No-skill')]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(ato_overall.model, ato_overall.ap_lift); axes[0].set_title('ATO AP lift'); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(ato_overall.model, ato_overall.roc_auc); axes[1].axhline(.5, color='black', linestyle='--'); axes[1].set_title('ATO ROC-AUC'); axes[1].tick_params(axis='x', rotation=20)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '02_ato_model_metrics.png', dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ato_overall.model, ato_overall.flag_rate * 100); ax.set(ylabel='Percent', title='Tỷ lệ login bị gắn cờ')
ax.tick_params(axis='x', rotation=20); fig.tight_layout()
fig.savefig(OUTPUT_DIR / '03_flag_rate.png', dpi=180); plt.close(fig)

if_temporal = temporal_metrics[(temporal_metrics.target == 'Is Account Takeover') & (temporal_metrics.model == 'B1 IsolationForest')]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(if_temporal.window, if_temporal.average_precision, marker='o'); axes[0].set_title('IF ATO Average Precision')
axes[1].plot(if_temporal.window, if_temporal.roc_auc, marker='o'); axes[1].axhline(.5, color='black', linestyle='--'); axes[1].set_title('IF ATO ROC-AUC')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / '04_temporal_stability.png', dpi=180); plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feature in zip(axes.flat, stability_features):
    ax.plot(monthly_features.index, monthly_features[f'{feature}_mean'], marker='o', label='mean')
    ax.plot(monthly_features.index, monthly_features[f'{feature}_median'], marker='.', label='median')
    ax.set_title(feature); ax.tick_params(axis='x', rotation=45); ax.legend()
fig.suptitle('Ổn định feature theo tháng'); fig.tight_layout()
fig.savefig(OUTPUT_DIR / '05_feature_stability.png', dpi=180); plt.close(fig)

rng = np.random.default_rng(RANDOM_STATE)
normal_scores = if_score[ato_truth.to_numpy() == 0]
normal_sample = rng.choice(normal_scores, size=min(200_000, len(normal_scores)), replace=False)
ato_scores = if_score[ato_truth.to_numpy() == 1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(normal_sample, bins=60, density=True, alpha=.65, label='Normal sample')
for value in ato_scores:
    ax.axvline(value, color='red', alpha=.35)
ax.set(title=f'IF score: normal và {len(ato_scores)} ATO', xlabel='Anomaly-oriented score', ylabel='Density')
ax.legend(); fig.tight_layout(); fig.savefig(OUTPUT_DIR / '06_error_analysis.png', dpi=180); plt.close(fig)
progress('figures: complete')

## 14 — Artifact thực nghiệm và metadata

`MinMaxScaler(clip=True)` chỉ chuẩn hóa score cho API `[0,1]`; nó không scale input và không biến score thành xác suất. Artifact không được tự động promote.

In [ ]:
progress('artifact export: start')
isolation_forest = models['B1 IsolationForest']
train_oriented_score = -isolation_forest.decision_function(X_train)
score_scaler = MinMaxScaler(clip=True).fit(train_oriented_score.reshape(-1, 1))
bundle = {
    'model': isolation_forest, 'score_scaler': score_scaler,
    'features': FEATURES, 'version': MODEL_VERSION,
    'score_input': '-decision_function',
    'feature_semantics_version': FEATURE_SEMANTICS_VERSION,
}
model_path = OUTPUT_DIR / f'{MODEL_VERSION}.joblib'
joblib.dump(bundle, model_path)

ato_if = benchmark[(benchmark.target == TARGETS[0]) & (benchmark.model == 'B1 IsolationForest')].iloc[0]
metadata = {
    'version': MODEL_VERSION, 'status': 'experimental', 'is_production': False,
    'algorithm': 'sklearn.ensemble.IsolationForest', 'features': FEATURES,
    'feature_semantics_version': FEATURE_SEMANTICS_VERSION,
    'training_labels_used': False, 'model_parameters': isolation_forest.get_params(),
    'normalization': 'MinMaxScaler(clip=True) fitted on training -decision_function',
    'score_semantics': 'higher means more anomalous; not a probability',
    'rows_loaded': int(len(model_data)), 'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)), 'cutoff_utc': cutoff.isoformat(),
    'test_ato_positives': int(y_test[TARGETS[0]].sum()),
    'ato_isolation_forest': {key: (None if pd.isna(ato_if[key]) else float(ato_if[key]))
                             for key in ['average_precision', 'ap_lift', 'roc_auc', 'precision', 'recall', 'f1', 'flag_rate']},
    'baseline_10m': BASELINE_10M,
    'python_version': platform.python_version(), 'sklearn_version': sklearn.__version__,
    'numpy_version': np.__version__, 'dataset': 'Wiefling et al., DOI 10.5281/zenodo.6782156',
    'dataset_warning': 'Synthetic research data; not for production IDS use',
}
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
progress(f'artifact export: complete {model_path.name}')

## 15 — Danh mục kết quả

In [ ]:
progress('final export inventory')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, path.stat().st_size)
heartbeat_stop.set()
progress('notebook complete')